In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import re
import joblib

/Users/dipanshuparashar/Desktop/project_sync/Log_Classification_NLP_BERT_LLM/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../dataset/synthetic_logs.csv')
display(df)

,timestamp,source,log_message,target_label
0,6/27/25 7:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status
1,1/14/25 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error
2,1/17/25 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert
3,7/12/25 0:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status
4,6/2/25 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status
...,...,...,...,...
2405,8/13/25 7:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status
2406,1/11/25 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert
2407,8/3/25 3:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status
2408,11/11/25 11:52,BillingSystem,Email service affected by failed transmission,Critical Error


In [3]:
df.source.unique()

<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [4]:
df.target_label.unique()

<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [5]:
# encode log messages to vectors
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = df['log_message'].astype(str).tolist()
embeddings = model.encode(texts)
embeddings[:2]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11704.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([[-1.02939703e-01,  3.35459337e-02, -2.20260508e-02,
         1.55098503e-03, -9.86916851e-03, -1.78956285e-01,
        -6.34410381e-02, -6.01762049e-02,  2.81108320e-02,
         5.99620081e-02, -1.72618385e-02,  1.43374118e-03,
        -1.49560049e-01,  3.15288105e-03, -5.66030778e-02,
         2.71685906e-02, -1.49890184e-02, -3.54038104e-02,
        -3.62936780e-02, -1.45410160e-02, -5.61498804e-03,
         8.75538513e-02,  4.55120578e-02,  2.50964370e-02,
         1.00187566e-02,  1.24266557e-02, -1.39923617e-01,
         7.68695921e-02,  3.14095281e-02, -4.15253639e-03,
         4.36902307e-02,  1.71250347e-02, -8.00950974e-02,
         5.74006215e-02,  1.89091638e-02,  8.55261534e-02,
         3.96398902e-02, -1.34371802e-01, -1.44367898e-03,
         3.06708645e-03,  1.76854104e-01,  4.44882922e-03,
        -1.69274919e-02,  2.24266555e-02, -4.35050316e-02,
         6.09036861e-03, -9.98167414e-03, -6.23972639e-02,
         1.07371677e-02, -6.04894478e-03, -7.14660510e-0

In [6]:
# DBSCAN clustering
dbscan = DBSCAN(eps=0.2, min_samples=1, metric='cosine')
clusters = dbscan.fit_predict(embeddings)
df['cluster'] = clusters
df.head()


,timestamp,source,log_message,target_label,cluster
0,6/27/25 7:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0
1,1/14/25 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,1
2,1/17/25 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,2
3,7/12/25 0:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,0
4,6/2/25 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,0


In [7]:
df[df.cluster == 1][['log_message', 'target_label']].head(20)

,log_message,target_label
1,Email service experiencing issues with sending,Critical Error
10,Email server encountered a sending fault,Error
217,Mail service encountered a delivery glitch,Error
248,Service disruption caused by email sending error,Critical Error
265,Email system had a problem sending emails,Error
361,Email service experienced a sending issue,Error
450,Email delivery system encountered an error,Error
477,Email transmission error caused service impact,Critical Error
570,Email service impacted by sending failure,Critical Error
678,Email delivery problem affected system,Critical Error


In [8]:
cluster_counts = df['cluster'].value_counts().sort_values(ascending=False)
print(cluster_counts)

for cluster_id, count in cluster_counts.items():
    if count > 10:
        print(f"\nCluster {cluster_id} (count {count})")
        for msg in df.loc[df['cluster'] == cluster_id, 'log_message'].head(5).tolist():
            print("-", msg)

cluster
0      1017
5       147
11      100
13       86
7        60
       ... 
99        1
100       1
102       1
103       1
135       1
Name: count, Length: 136, dtype: int64

Cluster 0 (count 1017)
- nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2675118
- nova.osapi_compute.wsgi.server [req-4895c258-b2f8-488f-a2a3-4fae63982e48 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" HTTP status code -  200 len: 211 time: 0.0968180
- nova.osapi_compute.wsgi.server [req-ee8bc8ba-9265-4280-9215-dbe000a41209 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" RCODE  200 len: 1874 time: 0.228

In [10]:
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.match(pattern, log_message, re.IGNORECASE):
            return label
    return None

In [12]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df[df.regex_label.notnull()]

,timestamp,source,log_message,target_label,cluster,regex_label
7,10/11/25 8:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,4,System Notification
14,1/4/25 1:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,4,System Notification
15,5/1/25 9:41,ModernCRM,Backup completed successfully.,System Notification,8,System Notification
18,2/22/25 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,9,User Action
27,9/24/25 19:57,ThirdPartyAPI,User User685 logged out.,User Action,11,User Action
...,...,...,...,...,...,...
2376,6/27/25 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,21,System Notification
2381,9/5/25 6:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,32,System Notification
2394,4/3/25 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,32,System Notification
2395,5/2/25 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,13,System Notification


In [13]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex.shape

(1910, 6)

In [14]:
small_labels = df_non_regex['target_label'].value_counts()
small_labels = small_labels[small_labels <= 5]

print("target_labels with 5 or fewer rows:")
print(small_labels)

# Optional: show rows for those labels
print("\nRows for those target_labels:")
print(df_non_regex[df_non_regex['target_label'].isin(small_labels.index)][['target_label', 'log_message']])

target_labels with 5 or fewer rows:
target_label
Workflow Error         4
Deprecation Warning    3
Name: count, dtype: int64

Rows for those target_labels:
             target_label                                        log_message
60         Workflow Error  Lead conversion failed for prospect ID 7842 du...
255   Deprecation Warning  API endpoint 'getCustomerDetails' is deprecate...
377        Workflow Error  Customer follow-up process for lead ID 5621 fa...
1325       Workflow Error  Escalation rule execution failed for ticket ID...
1734  Deprecation Warning  The 'ExportToCSV' feature is outdated. Please ...
1826  Deprecation Warning  Support for legacy authentication methods will...
2217       Workflow Error  Task assignment for TeamID 3425 could not comp...


In [15]:
df_non_legacy=df_non_regex[df_non_regex.source!='LegacyCRM']
df_non_legacy.source.unique()

<StringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [16]:
texts_non_legacy = df_non_legacy['log_message'].astype(str).tolist()
filtered_embeddings = model.encode(texts_non_legacy)
filtered_embeddings[:2]

array([[-1.02939703e-01,  3.35459337e-02, -2.20260508e-02,
         1.55098503e-03, -9.86916851e-03, -1.78956285e-01,
        -6.34410381e-02, -6.01762049e-02,  2.81108320e-02,
         5.99620081e-02, -1.72618385e-02,  1.43374118e-03,
        -1.49560049e-01,  3.15288105e-03, -5.66030778e-02,
         2.71685906e-02, -1.49890184e-02, -3.54038104e-02,
        -3.62936780e-02, -1.45410160e-02, -5.61498804e-03,
         8.75538513e-02,  4.55120578e-02,  2.50964370e-02,
         1.00187566e-02,  1.24266557e-02, -1.39923617e-01,
         7.68695921e-02,  3.14095281e-02, -4.15253639e-03,
         4.36902307e-02,  1.71250347e-02, -8.00950974e-02,
         5.74006215e-02,  1.89091638e-02,  8.55261534e-02,
         3.96398902e-02, -1.34371802e-01, -1.44367898e-03,
         3.06708645e-03,  1.76854104e-01,  4.44882922e-03,
        -1.69274919e-02,  2.24266555e-02, -4.35050316e-02,
         6.09036861e-03, -9.98167414e-03, -6.23972639e-02,
         1.07371677e-02, -6.04894478e-03, -7.14660510e-0

In [17]:
X = filtered_embeddings
y = df_non_legacy['target_label']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize and train the Logistic Regression model
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Generate and print the classification report
report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [19]:
import joblib

joblib.dump(clf,'../models/log_classifier.joblib')




['../models/log_classifier.joblib']